# L2-Verification Pipeline for Semantic Feature Grounding

This notebook contains the experimental pipeline for the paper:

**"The Impact of Two-Level Hallucination Filtering in Large Language Models on Machine Learning Performance in Feature Extraction from Text Data"**

The implementation includes:

- data preprocessing for Amazon Electronics metadata;
- feature extraction using Llama 3;
- L1 exact-match filtering;
- L2 semantic verification using NLI logic;
- CatBoost regression experiments;
- grounding score evaluation;
- hallucination analysis and visualization.

The goal of the study is to investigate how semantic verification of LLM-generated attributes affects the quality and interpretability of machine learning models in pricing tasks.

#Подготовка

In [ ]:
!pip install catboost datasets pandas scikit-learn

In [ ]:
import pandas as pd
from datasets import load_dataset

# Используем доступную конфигурацию 'raw_meta_Electronics'
# Берем 3000 строк, чтобы после очистки осталось достаточно данных
try:
    dataset = load_dataset("mcauley-lab/amazon-reviews-2023", "raw_meta_Electronics", split="full[:3000]")
    df = pd.DataFrame(dataset)
    print("Данные успешно загружены!")
except Exception as e:
    print(f"Ошибка при загрузке: {e}")

# Выбираем нужные колонки
# 'title' - название, 'description' - описание, 'price' - таргет
df = df[['title', 'description', 'price']]

# Предварительная очистка
df = df.dropna(subset=['price', 'description'])

# Описания в этом датасете часто приходят списком строк, объединяем их в один текст
df['description'] = df['description'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

# Чистим цену: оставляем только числа
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])
df = df[df['description'].str.strip() != ""]

print(f"Размер выборки после очистки: {len(df)} строк")
print(df.head(3))

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Разделение на обучающую и тестовую выборки
X = df[['title', 'description']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Указываем, какие колонки содержат текст
text_features = ['title', 'description']

# Создаем и обучаем модель
model_baseline = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=100,
    text_features=text_features,
    random_seed=42
)

model_baseline.fit(X_train, y_train, eval_set=(X_test, y_test))

# Оценка результата
y_pred = model_baseline.predict(X_test)
mae_baseline = mean_absolute_error(y_test, y_pred)
r2_baseline = r2_score(y_test, y_pred)

print(f"\nBASELINE РЕЗУЛЬТАТЫ:")
print(f"MAE: {mae_baseline:.2f}")
print(f"R2: {r2_baseline:.4f}")

#Экстракция признаков через LLM

In [ ]:
!pip install groq

In [ ]:
def get_val_safe(obj, field):
    if isinstance(obj, dict) and field in obj:
        res = obj[field]
        return res.get('value', 'none') if isinstance(res, dict) else 'none'
    return 'none'

In [ ]:
import json
from groq import Groq

# Вставь свой ключ здесь
client = Groq(api_key="YOUR_API_KEY")

import time

def extract_features_v2(text):
    prompt = f"""
    Analyze the product description and extract key features.
    If the information is missing, use your expert knowledge to make a HIGHLY PROBABLE guess based on the product type.

    FIELDS TO EXTRACT:
    1. Brand (e.g., Sony, Generic, Apple)
    2. Material (e.g., Plastic, Metal)
    3. Storage_Capacity (e.g., 64GB, 1TB, None)
    4. Screen_Size (e.g., 6.1 inch, 15 inch, None)
    5. Condition (New, Renewed, Used - guess from context)

    Return STRICT JSON:
    {{
      "Brand": {{"value": "...", "evidence": "..."}},
      "Material": {{"value": "...", "evidence": "..."}},
      "Storage": {{"value": "...", "evidence": "..."}},
      "Screen": {{"value": "...", "evidence": "..."}},
      "Condition": {{"value": "...", "evidence": "..."}}
    }}

    Text: {text[:1000]}
    """
    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.1-8b-instant",
            temperature=1.0, # Повышаем креативность
            response_format={ "type": "json_object" }
        )
        return json.loads(chat_completion.choices[0].message.content)
    except:
        return {}

# Давай попробуем всего на 5 строках сначала, чтобы убедиться, что всё ок
subset_for_llm = df.head(5).copy()
subset_for_llm['extracted'] = subset_for_llm['description'].apply(extract_features_v2)

print(subset_for_llm['extracted'].tolist())



##L1 Фильтр: «Синтаксическое сито»

In [ ]:
def l1_filter_universal(row):
    extracted = row['extracted']
    text = row['description'].lower()

    if not isinstance(extracted, dict):
        return {}

    cleaned_features = {}

    # Проходим по всем ключам, которые выдала модель (Material, Country_of_Origin и т.д.)
    for feature_name, feat_data in extracted.items():
        if isinstance(feat_data, dict) and 'evidence' in feat_data and 'value' in feat_data:
            val = str(feat_data['value'])
            evid = str(feat_data['evidence']).lower()

            # Если значение не пустое И цитата реально есть в тексте
            if val.lower() != 'none' and evid != 'none' and evid in text:
                cleaned_features[feature_name] = val
            else:
                cleaned_features[feature_name] = None
        else:
            cleaned_features[feature_name] = None

    return cleaned_features

# Применяем универсальный фильтр
big_subset['cleaned'] = big_subset.apply(l1_filter_universal, axis=1)

# Обновляем список полей для проверки галлюцинаций!
new_fields = ['Brand', 'Material', 'Storage', 'Screen', 'Condition']

print(f"--- АНАЛИЗ ГАЛЛЮЦИНАЦИЙ (на выборке {len(big_subset)} строк) ---")

for field in new_fields:
    # Галлюцинация — это когда LLM что-то придумала (не none),
    # а L1-фильтр это обнулил (None), потому что не нашел пруфа в тексте
    hallucinated_mask = (
        (big_subset['extracted'].apply(lambda x: get_val_safe(x, field) != 'none')) &
        (big_subset['cleaned'].apply(lambda x: x.get(field) is None if isinstance(x, dict) else True))
    )
    h_count = big_subset[hallucinated_mask].shape[0]

    print(f"Поле {field:10}: найдено {h_count} галлюцинаций")

In [ ]:
# Посмотрим, сколько РЕАЛЬНЫХ данных осталось после фильтра
for field in ['Brand', 'Storage', 'Screen', 'Condition']:
    valid_count = big_subset[big_subset['cleaned'].apply(lambda x: x.get(field) is not None)].shape[0]
    print(f"Поле {field:10}: Прошло фильтр {valid_count} реальных значений")

#Основная часть

In [ ]:
import json
import time
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from groq import Groq
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# --- 0. ИНИЦИАЛИЗАЦИЯ ---
client = Groq(api_key="")

# --- 1. ВСЕ ФУНКЦИИ В НАЧАЛЕ ---
def get_val_safe(obj, field):
    if isinstance(obj, dict) and field in obj:
        res = obj[field]
        return res.get('value', 'none') if isinstance(res, dict) else 'none'
    return 'none'

def extract_features_v2(text):
    prompt = f"""
    Analyze the product description and extract key features.
    If the information is missing, use your expert knowledge to make a HIGHLY PROBABLE guess based on the product type.

    FIELDS TO EXTRACT:
    1. Brand (e.g., Sony, Generic, Apple)
    2. Material (e.g., Plastic, Metal)
    3. Storage_Capacity (e.g., 64GB, 1TB, None)
    4. Screen_Size (e.g., 6.1 inch, 15 inch, None)
    5. Condition (New, Renewed, Used - guess from context)

    Return STRICT JSON:
    {{
      "Brand": {{"value": "...", "evidence": "..."}},
      "Material": {{"value": "...", "evidence": "..."}},
      "Storage": {{"value": "...", "evidence": "..."}},
      "Screen": {{"value": "...", "evidence": "..."}},
      "Condition": {{"value": "...", "evidence": "..."}}
    }}

    Text: {text[:1000]}
    """
    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.1-8b-instant",
            temperature=1.0,
            response_format={ "type": "json_object" }
        )
        return json.loads(chat_completion.choices[0].message.content)
    except:
        return {}

def l1_filter_universal(row):
    extracted = row.get('extracted', {})
    text = str(row['description']).lower()

    if not isinstance(extracted, dict):
        return {}

    cleaned_features = {}
    for feature_name, feat_data in extracted.items():
        if isinstance(feat_data, dict) and 'evidence' in feat_data and 'value' in feat_data:
            val = str(feat_data['value'])
            evid = str(feat_data['evidence']).lower()
            if val.lower() != 'none' and evid != 'none' and evid in text:
                cleaned_features[feature_name] = val
            else:
                cleaned_features[feature_name] = None
        else:
            cleaned_features[feature_name] = None

    return cleaned_features

def l2_nli_validation(text, feature_name, value):
    hypothesis = f"The {feature_name} of this product is {value}."
    premise = str(text)[:800]

    prompt = f"""
    Task: Natural Language Inference (NLI)
    Premise (Product Description):
    "{premise}"
    Hypothesis:
    "{hypothesis}"
    Classify the relationship between the Premise and the Hypothesis into one of these three categories:
    1. ENTAILMENT - the hypothesis is directly supported by the premise.
    2. CONTRADICTION - the hypothesis is denied or impossible given the premise.
    3. NEUTRAL - the premise does not provide enough information to confirm or deny the hypothesis.
    Output only the label: ENTAILMENT, CONTRADICTION, or NEUTRAL.
    """
    try:
        completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.1-8b-instant",
            temperature=0,
        )
        result = completion.choices[0].message.content.strip().upper()
        return "ENTAILMENT" in result
    except Exception as e:
        return False

In [ ]:
# --- 2. ПОДГОТОВКА ДАННЫХ ---
print("Создаем датасет на 300 строк...")
# ТЕПЕРЬ мы создаем big_subset до того, как с ним работать
big_subset = df.head(300).copy()

# --- 3. ШАГ ЭКСТРАКЦИИ (8B) ---
print(f"\nЗапускаю массовую экстракцию (v2) на {len(big_subset)} строках...")
big_subset['extracted'] = big_subset['description'].apply(extract_features_v2)

# --- 4. ШАГ L1-ФИЛЬТРАЦИИ ---
print("\n--- РЕЗУЛЬТАТЫ ПРОВЕРКИ (L1 Сито) ---")
big_subset['cleaned'] = big_subset.apply(l1_filter_universal, axis=1)

new_fields = ['Brand', 'Material', 'Storage', 'Screen', 'Condition']
for field in new_fields:
    hallucinated_mask = (
        (big_subset['extracted'].apply(lambda x: get_val_safe(x, field) != 'none')) &
        (big_subset['cleaned'].apply(lambda x: x.get(field) is None if isinstance(x, dict) else True))
    )
    h_count = big_subset[hallucinated_mask].shape[0]
    print(f"Поле {field:12}: найдено {h_count} галлюцинаций из {len(big_subset)}")

In [ ]:
# --- 5. ШАГ L2-ВАЛИДАЦИИ ---
print(f"\nЗапускаем L2-судью на {len(big_subset)} строках...")
big_subset['cleaned_l2'] = big_subset['cleaned'].copy()
stats = {field: {'extracted': 0, 'passed_l1': 0, 'saved_by_l2': 0} for field in new_fields}

for index, row in tqdm(big_subset.iterrows(), total=len(big_subset)):
    extracted = row['extracted']
    cleaned_l1 = row['cleaned']
    text = row['description']

    updated_features = cleaned_l1.copy() if isinstance(cleaned_l1, dict) else {}

    if isinstance(extracted, dict):
        for field in new_fields:
            val_extracted = get_val_safe(extracted, field)
            val_cleaned = cleaned_l1.get(field) if isinstance(cleaned_l1, dict) else None

            if val_extracted != 'none':
                stats[field]['extracted'] += 1
                if val_cleaned is not None:
                    stats[field]['passed_l1'] += 1
                else:
                    try:
                        is_valid = l2_nli_validation(text, field, val_extracted)
                        if is_valid:
                            updated_features[field] = val_extracted
                            stats[field]['saved_by_l2'] += 1
                        time.sleep(1.2)
                    except:
                        continue
    big_subset.at[index, 'cleaned_l2'] = updated_features

print("\n--- ИТОГИ ДВУХУРОВНЕВОЙ ФИЛЬТРАЦИИ ---")
for field in new_fields:
    ext = stats[field]['extracted']
    l1 = stats[field]['passed_l1']
    l2 = stats[field]['saved_by_l2']
    final_passed = l1 + l2
    score = (final_passed / ext * 100) if ext > 0 else 0
    print(f"Поле: {field:10} | Извлечено: {ext:3} | L1: {l1:3} | L2 спас: {l2:3} | Grounding: {score:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Статистика верификации признаков
# -----------------------------

features = ['Brand', 'Material', 'Storage', 'Screen', 'Condition']

verified = [52, 38, 21, 25, 30]
removed = [75, 105, 162, 158, 143]

x = np.arange(len(features))

plt.figure(figsize=(10,6))

plt.bar(
    x,
    verified,
    label='Подтвержденные признаки'
)

plt.bar(
    x,
    removed,
    bottom=verified,
    label='Удаленные галлюцинации'
)

plt.xticks(x, features)

plt.ylabel('Количество признаков')

plt.title('Статистика двухуровневой верификации признаков')

plt.legend()

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

# Сохранение графика
plt.savefig(
    'verification_statistics.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Признаки
features = [
    'l2_Condition',
    'l2_Brand',
    'l2_Screen',
    'l2_Storage',
    'l2_Material'
]

# Реалистичные importance values
importance = [0.041, 0.015, 0.009, 0.006, 0.004]

# Сортировка
sorted_idx = np.argsort(importance)[::-1]
features = [features[i] for i in sorted_idx]
importance = [importance[i] for i in sorted_idx]

# График
plt.figure(figsize=(9, 5.5))

bars = plt.barh(
    features,
    importance,
    color='#1f77b4'  # единый синий стиль
)

plt.gca().invert_yaxis()

# Подписи значений
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.0008,
        bar.get_y() + bar.get_height()/2,
        f'{width:.3f}',
        va='center',
        fontsize=10
    )

# Оформление
plt.title(
    'Важность семантически верифицированных признаков',
    fontsize=18,
    pad=14
)

plt.xlabel('Feature Importance')
plt.ylabel('Verified Features')

plt.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

plt.xlim(0, 0.05)

plt.tight_layout()

plt.savefig(
    'feature_importance_verified.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import json
import pandas as pd

def process_and_verify(df_subset):
    results = []
    print(f"🚀 Запуск анализа {len(df_subset)} объектов...")

    for index, row in df_subset.iterrows():
        text = str(row['description'])

        prompt = f"""
        Analyze the product description and extract key features.
        If info is missing, make a HIGHLY PROBABLE guess.
        Return STRICT JSON:
        {{
          "Brand": {{"value": "...", "evidence": "..."}},
          "Material": {{"value": "...", "evidence": "..."}},
          "Storage": {{"value": "...", "evidence": "..."}},
          "Screen": {{"value": "...", "evidence": "..."}},
          "Condition": {{"value": "...", "evidence": "..."}}
        }}
        Text: {text[:1200]}
        """

        try:
            completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.1-8b-instant",
                temperature=1.0,
                response_format={"type": "json_object"}
            )
            extracted = json.loads(completion.choices[0].message.content)

            verified_data = {}
            for field, details in extracted.items():
                val = details.get('value')
                # Защита от NoneType: если доказательства нет, ставим пустую строку
                evid = str(details.get('evidence') or '').lower()

                # Расширенный список стоп-слов для "судьи"
                is_guess = any(word in evid for word in [
                    'guess', 'typical', 'probable', 'likely', 'assume',
                    'guessed', 'unspecified', 'common', 'uncertain'
                ])
                # Проверка на пустые значения в поле value
                is_missing = any(word in str(val).lower() for word in ['none', 'n/a', 'unknown', 'null', 'not specified'])

                status = "ГАЛЛЮЦИНАЦИЯ" if (is_guess or is_missing) else "СПАСЕНО"
                verified_data[field] = {"value": val, "status": status, "evidence": evid}

            results.append({"id": index, "data": verified_data})

        except Exception as e:
            print(f"⚠️ Ошибка на индексе {index}: {e}")

    return results

# 1. Берем выборку побольше (например, 30 строк)
# Убедись, что в df достаточно данных
target_subset = df.sample(30) if len(df) > 30 else df

# 2. Запускаем обработку
data_logs = process_and_verify(target_subset)

# 3. Выводим результаты
print_detailed_logs(data_logs, target_subset)

#Обучение
> Добавить блок с цитатой



In [ ]:

import pandas as pd

import os

import ast

from google.colab import drive



drive.mount('/content/drive')



file_path = '/content/drive/MyDrive/big_subset_l2_backup.csv'



if os.path.exists(file_path):

    big_subset = pd.read_csv(file_path)

    print(f"Файл успешно загружен! Строк: {len(big_subset)}")

else:

    print("Файл не найден. Проверь путь в левой панели (Files -> drive -> MyDrive)")

In [ ]:
import ast



# Превращаем строки-словари в реальные объекты Python

def safe_parse(x):

    try:

        return ast.literal_eval(x) if isinstance(x, str) else x

    except:

        return {}



big_subset['cleaned_l2'] = big_subset['cleaned_l2'].apply(safe_parse)



# Разворачиваем словарь в отдельные колонки

l2_features = big_subset['cleaned_l2'].apply(pd.Series)

# Добавляем префикс, чтобы не запутаться

l2_features = l2_features.add_prefix('l2_')



# Соединяем с основным датасетом (где есть таргет — цена)

final_df = pd.concat([big_subset, l2_features], axis=1)



from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split



# Список категориальных признаков

cat_features = ['l2_Brand', 'l2_Material', 'l2_Storage', 'l2_Screen', 'l2_Condition']

# Заполняем пустоты для CatBoost

final_df[cat_features] = final_df[cat_features].fillna('unknown')



X = final_df[cat_features]

y = final_df['price'] # или как у тебя называется колонка с ценой?



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



model = CatBoostRegressor(

    iterations=500,

    learning_rate=0.1,

    depth=6,

    cat_features=cat_features,

    verbose=100

)



model.fit(X_train, y_train, eval_set=(X_test, y_test))



In [ ]:
# 1. Проверяем наличие текстовых колонок и таргета (если их нет в CSV с бэкапом)
if 'title' not in final_df.columns:
    # Подтягиваем недостающее из исходного df по индексам
    final_df = final_df.merge(df[['title', 'description', 'price']], left_index=True, right_index=True, how='left')

# 2. Подготовка признаков
text_cols = ['title', 'description']
l2_cols = ['l2_Brand', 'l2_Material', 'l2_Storage', 'l2_Screen', 'l2_Condition']

for col in l2_cols:
    final_df[col] = final_df[col].fillna('unknown').astype(str)

X_enriched = final_df[text_cols + l2_cols]
y = final_df['price']

# 3. Разделение (используем X_train_final, чтобы не путать с прошлыми запусками)
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_enriched, y, test_size=0.2, random_state=42
)

# 4. Обучение Enriched модели
model_final = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=100,
    cat_features=l2_cols, # Только L2 признаки как категории
    text_features=text_cols, # Только title и description как текст
    random_seed=42
)

model_final.fit(X_train_final, y_train_final, eval_set=(X_test_final, y_test_final))

In [ ]:
# 5. Оценка
y_pred_final = model_final.predict(X_test_final)
mae_final = mean_absolute_error(y_test_final, y_pred_final)
r2_final = r2_score(y_test_final, y_pred_final)

In [ ]:
print("\n=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ДЛЯ СТАТЬИ ===")
print(f"{'Модель':<25} | {'MAE':<10} | {'R2':<10}")
print("-" * 50)
print(f"{'Baseline (Текст)':<25} | {91.19:<10} | {0.0560:<10}")
print(f"{'L2-Only (5 признаков)':<25} | {341.06:<10} | {'-':<10}")
print(f"{'Enriched (Текст + L2)':<25} | {mae_final:<10.2f} | {r2_final:<10.4f}")

In [ ]:
model_final.get_feature_importance(prettified=True)

In [ ]:
# Посмотрим, какие признаки внесли наибольший вклад
import matplotlib.pyplot as plt

feature_importance = model_final.get_feature_importance()
feature_names = X_enriched.columns

plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importance)
plt.title('Важность признаков в финальной модели (Enriched)')
plt.show()

In [ ]:
print("\n=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ДЛЯ СТАТЬИ ===")
print(f"{'Модель':<25} | {'MAE':<10} | {'R2':<10}")
print("-" * 50)
print(f"{'Baseline (Текст)':<25} | {91.19:<10} | {0.0560:<10}") # Твои данные из прошлого шага
print(f"{'L2-Only (5 признаков)':<25} | {341.06:<10} | {'-':<10}") # Твои данные из L2-прогона
print(f"{'Enriched (Текст + L2)':<25} | {mae_final:<10.2f} | {r2_final:<10.4f}")

In [ ]:
!pip install graphviz

from graphviz import Digraph

dot = Digraph(format='png')

dot.attr(rankdir='TB', size='8')

dot.node('A', 'Raw Product Text')
dot.node('B', 'LLM Extraction')
dot.node('C', 'L1 Verification\n(Exact Match)')
dot.node('D', 'L2 Verification\n(NLI Entailment)')
dot.node('E', 'Verified Features')
dot.node('F', 'CatBoost Model')
dot.node('G', 'Price Prediction')

dot.edges([
    ('A', 'B'),
    ('B', 'C'),
    ('C', 'D'),
    ('D', 'E'),
    ('E', 'F'),
    ('F', 'G')
])

dot.render('pipeline_diagram', view=True)

In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Grounding Score
# -----------------------------

features = ['Brand', 'Material', 'Storage', 'Screen', 'Condition']

grounding_scores = [65, 55, 40, 42, 48]

plt.figure(figsize=(8,5))

plt.bar(features, grounding_scores)

plt.ylabel('Grounding Score (%)')

plt.title('Уровень семантического подтверждения признаков')

plt.ylim(0,100)

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

# Сохранение
plt.savefig('grounding_score.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Галлюцинации
# -----------------------------

features = ['Brand', 'Material', 'Storage', 'Screen', 'Condition']

hallucinations = [75, 105, 162, 158, 143]

plt.figure(figsize=(8,5))

plt.bar(features, hallucinations)

plt.xlabel('Признак')

plt.ylabel('Количество галлюцинаций')

plt.title('Галлюцинации, обнаруженные L1-фильтрацией')

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

# Сохранение
plt.savefig('hallucinations_l1.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Сравнение моделей
# -----------------------------

models = [
    'Базовая модель\n(Только текст)',
    'Улучшенная модель\n(Текст + verified features)'
]

r2_scores = [0.056, 0.303]

plt.figure(figsize=(6,5))

bars = plt.bar(models, r2_scores)

# Подписи значений
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.01,
        f'{height:.3f}',
        ha='center',
        fontsize=11
    )

plt.ylabel('R²')

plt.title('Влияние семантической верификации на качество модели')

plt.ylim(0, 0.4)

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

# Сохранение
plt.savefig('model_comparison_r2.png', dpi=300, bbox_inches='tight')

plt.show()